# Carlos OS Puro: verificação no Colab

Este notebook **não compila nada**. Ele conecta o Drive, baixa o projeto e verifica se o Colab serve para o Linux From Scratch.

Rode as células na ordem e cole o resultado da última no Codex, junto com `docs/prompt-codex-puro.md`.

## 1. Conectar o Google Drive

O Google pede sua autorização na tela. O progresso futuro fica em `Meu Drive/Carlos-OS-Puro`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/Carlos-OS-Puro'
!ls -ld '/content/drive/MyDrive/Carlos-OS-Puro'

## 2. Baixar o projeto

In [ ]:
%cd /content
!test -d carlos-ia || git clone https://github.com/eduardoc7417-hub/carlos-ia.git
!git -C carlos-ia log --oneline -1
!ls carlos-ia/docs

## 3. Verificar a máquina

Só o teste de `mount` altera algo, e ele desmonta em seguida. Cole esta saída no Codex.

In [ ]:
%%bash
echo '--- maquina'
nproc
grep -m1 'model name' /proc/cpuinfo
free -h
df -h /content /tmp
uname -r
id
echo '--- ferramentas'
for t in bash gcc g++ ld bison gawk m4 make patch perl python3 sed tar makeinfo xz qemu-system-x86_64; do
  printf '%s: ' "$t"; command -v "$t" >/dev/null && "$t" --version 2>/dev/null | head -1 | cut -c1-60 || echo ausente
done
echo "sh -> $(readlink -f /bin/sh)"
echo '--- permissoes'
ls -l /dev/kvm 2>&1
ls -l /dev/loop0 /dev/loop-control 2>&1
mkdir -p /tmp/teste-mount
if mount -t tmpfs tmpfs /tmp/teste-mount 2>&1; then echo 'mount: OK'; umount /tmp/teste-mount; else echo 'mount: FALHOU'; fi
if chroot / /bin/true 2>&1; then echo 'chroot: OK'; else echo 'chroot: FALHOU'; fi
truncate -s 64M /tmp/teste.img
if command -v losetup >/dev/null && L=$(losetup -f --show /tmp/teste.img 2>&1); then echo "loop: OK $L"; losetup -d "$L"; else echo "loop: FALHOU"; fi
rm -f /tmp/teste.img
